### Fonctions de qualité des données — projet Ventes

Principe directeur : quarantaine plutôt que suppression**

Chaque fonction de nettoyage ci-dessous retourne un **tuple** `(df_valide, df_rejeté)
plutôt qu'un DataFrame unique déjà filtré. Pourquoi : si on filtre et jette silencieusement les lignes invalides, on perd toute trace de ce qui a été exclu et pourquoi — impossible de répondre à la question "pourquoi le CA de mars est-il plus bas que prévu, a-t-on perdu des ventes en route ?". En gardant les rejets dans une table séparée (`silver.ventes.ventes_rejected` par exemple), on peut auditer, corriger la source, et rejouer le pipeline.

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
# MAGIC On a injecté volontairement `Paris` / `PARIS` / `paris` dans les données
# MAGIC sources. `F.initcap()` met en majuscule la première lettre de chaque mot et
# MAGIC force le reste en minuscule — exactement ce qu'il faut ici. `F.trim()` en
# MAGIC plus au cas où des espaces parasites traînent (fréquent avec des exports CSV).
# MAGIC 
# MAGIC Cette fonction ne rejette rien : la standardisation de casse n'invalide
# MAGIC jamais une ligne, elle la corrige simplement.

def standardize_text_column(df: DataFrame, column: str) -> DataFrame:
    """
    Standardise la casse d'une colonne texte (trim + initcap).
    Utilisé pour 'region' mais réutilisable pour toute colonne texte
    sujette à des variantes de saisie.
    """
    return df.withColumn(column, F.initcap(F.trim(F.col(column))))

In [0]:
# MAGIC Rappel du choix fait dans `schema_definitions.py` : les dates arrivent en
# MAGIC `StringType`. On les caste ici en `DateType`, mais **en comptant les échecs**
# MAGIC plutôt qu'en laissant `to_date()` transformer silencieusement une date
# MAGIC mal formée en `null`.
# MAGIC 
# MAGIC La logique : une ligne où `date_vente` (string) n'était pas nulle au départ,
# MAGIC mais dont le cast donne `null`, est une ligne dont le format de date était
# MAGIC invalide → elle part en quarantaine plutôt que d'entrer silencieusement
# MAGIC avec une date manquante.

def safe_cast_date(
    df: DataFrame,
    column: str,
    date_format: str = "yyyy-MM-dd"
) -> tuple[DataFrame, DataFrame]:
    """
    Caste une colonne string en date, isole les échecs de cast.
    
    Returns:
        (df_valide, df_rejete) où df_rejete contient les lignes
        dont la date source était non-nulle mais n'a pas pu être castée.
    """
    df_casted = df.withColumn(
        f"{column}_casted",
        F.to_date(F.col(column), date_format)
    )

    cast_failed = (
        F.col(column).isNotNull() & F.col(f"{column}_casted").isNull()
    )

    df_rejete = df_casted.filter(cast_failed).drop(f"{column}_casted")
    df_valide = (
        df_casted.filter(~cast_failed)
        .drop(column)
        .withColumnRenamed(f"{column}_casted", column)
    )

    return df_valide, df_rejete

In [0]:
# MAGIC Règle métier : une quantité vendue ne peut pas être négative ou nulle.
# MAGIC On sait avoir injecté des quantités négatives dans les données sources —
# MAGIC cette fonction les isole plutôt que de les laisser fausser des agrégations
# MAGIC plus tard (`SUM(quantite)` serait faux si des négatifs restent mélangés
# MAGIC sans qu'on le sache).

def filter_invalid_quantities(
    df: DataFrame,
    column: str = "quantite"
) -> tuple[DataFrame, DataFrame]:
    """
    Isole les lignes où la quantité est négative, nulle, ou zéro.
    
    Returns:
        (df_valide, df_rejete)
    """
    is_invalid = F.col(column).isNull() | (F.col(column) <= 0)

    df_rejete = df.filter(is_invalid)
    df_valide = df.filter(~is_invalid)

    return df_valide, df_rejete

In [0]:
# MAGIC `dropDuplicates()` sans argument compare **toutes** les colonnes — une ligne
# MAGIC n'est considérée doublon que si elle est identique en tout point à une autre.
# MAGIC C'est volontairement strict : on ne veut pas risquer de fusionner deux ventes
# MAGIC différentes qui se ressembleraient par coïncidence (même produit, même jour,
# MAGIC quantité différente par exemple — ce n'est PAS un doublon).
# MAGIC 
# MAGIC Contrairement aux fonctions précédentes, celle-ci ne retourne qu'un seul
# MAGIC DataFrame : un doublon exact n'est pas une "erreur à auditer", c'est une
# MAGIC répétition à éliminer silencieusement (la ligne d'origine reste présente une fois).

def remove_exact_duplicates(df: DataFrame) -> DataFrame:
    """Supprime les lignes strictement identiques sur toutes les colonnes."""
    return df.dropDuplicates()

In [0]:
# MAGIC Utilisé pour `retours` : certains `id_vente` référencent une vente qui
# MAGIC n'existe pas dans `ventes` (14 lignes injectées volontairement). On utilise
# MAGIC un **left anti join** — il retourne uniquement les lignes de `df` qui n'ont
# MAGIC AUCUNE correspondance dans `df_reference` sur la clé donnée.

def flag_orphan_records(
    df: DataFrame,
    df_reference: DataFrame,
    join_key: str
) -> tuple[DataFrame, DataFrame]:
    """
    Sépare les lignes de df qui ont une correspondance dans df_reference
    de celles qui n'en ont pas (orphelines).
    
    Returns:
        (df_valide, df_orphelins)
    """
    df_orphelins = df.join(df_reference, on=join_key, how="left_anti")
    df_valide = df.join(
        df_reference.select(join_key).distinct(),
        on=join_key,
        how="left_semi"
    )

    return df_valide, df_orphelins

In [0]:
# MAGIC Fonction simple mais essentielle : sans elle, on lance des fonctions de
# MAGIC nettoyage à l'aveugle sans savoir combien de lignes sont parties en
# MAGIC quarantaine. `.count()` déclenche un calcul complet sur le DataFrame — coûteux
# MAGIC sur un très gros volume, mais parfaitement raisonnable ici et bien plus
# MAGIC utile en apprentissage qu'une optimisation prématurée.

def log_quality_summary(step_name: str, df_valide: DataFrame, df_rejete: DataFrame) -> None:
    """Affiche un résumé lisible du nombre de lignes valides vs rejetées."""
    n_valide = df_valide.count()
    n_rejete = df_rejete.count()
    total = n_valide + n_rejete
    pct_rejete = (n_rejete / total * 100) if total > 0 else 0

    print(f"[{step_name}] {n_valide} lignes valides, "
          f"{n_rejete} rejetées ({pct_rejete:.1f}%) sur {total} au total")